# Cyprus Terrace Detection - Random Forest (Submission)
Minimal workflow: prepare data, train binary `terrace` model, and report validation metrics.

## 1. Data Preparation

In [ ]:
import os
import pandas as pd

# ---------- Config ----------
OUTPUT_CSV = globals().get("OUTPUT_CSV", r"E:/Cyprus_paper_data/polygons_sample_points.csv")

valid_grids = [11, 13, 21, 25, 2, 30, 34, 38, 44, 51, 52, 56, 6, 8]
train_grids = [10, 12, 14, 15, 16, 17, 18, 19, 1, 20, 22, 23, 24, 26, 27, 28, 29, 31, 32, 33, 35, 36, 37, 39, 3, 40, 41, 42, 43, 45, 46, 47, 48, 49, 4, 50, 53, 54, 55, 5, 7, 9]

cols_to_drop = [
    "name", "mean_red", "mean_green", "mean_blue", "mean_grayscale",
    "range_elevation", "std_elevation", "percentile_10_slope", "percentile_90_slope",
    "mean_slope", "std_slope", "range_slope", "range_profcurv", "std_profcurv",
    "range_plancurv", "std_plancurv", "majority_landcover", "mean_mean_text",
    "std_mean_text", "mean_std_text", "std_std_text", "mean_cont_text", "std_cont_text",
    "mean_homo_text", "std_homo_text", "mean_energy_text", "std_energy_text",
    "mean_clsh_text", "std_clsh_text", "mean_entropy_text", "std_entropy_text",
    "mean_canny_200_400", "mean_edges_200_400", "mean_canny_150_300",
    "mean_edges_150_300", "mean_canny_100_250", "mean_edges_100_250",
    "dem_clipped", "flowdir", "flowdir_res", "grayscale", "inflated_dem", "red",
    "flowdir_resampled_edges"
]

# ---------- Load + preprocess ----------
if not OUTPUT_CSV or not os.path.exists(OUTPUT_CSV):
    raise FileNotFoundError(f"Saved CSV not found: {OUTPUT_CSV}")

df = pd.read_csv(OUTPUT_CSV)
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

required_cols = ["terrace", "gridnumber"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[df["terrace"].isin([0, 1])].copy()

train_df = df[df["gridnumber"].isin(train_grids)].copy()
valid_df = df[df["gridnumber"].isin(valid_grids)].copy()

if "selected" in valid_df.columns:
    valid_df = valid_df[valid_df["selected"] == 1].copy()

drop_meta_cols = ["ancient", "gridnumber", "source_poly_id", "selected"]
train_df = train_df.drop(columns=[c for c in drop_meta_cols if c in train_df.columns], errors="ignore")
valid_df = valid_df.drop(columns=[c for c in drop_meta_cols if c in valid_df.columns], errors="ignore")

train_df = train_df.dropna()
valid_df = valid_df.dropna()

print(f"Train rows: {len(train_df)}")
print(f"Validation rows: {len(valid_df)}")
print("Validation class distribution:")
print(valid_df["terrace"].value_counts(dropna=False))

## 2. Binary Random Forest Training
Train a binary classifier for `terrace` using the prepared train/validation split.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

label = "terrace"
if label not in train_df.columns or label not in valid_df.columns:
    raise ValueError("Missing 'terrace' label in train/validation dataframe.")

y_train = train_df[label].astype(int)
X_train = train_df.drop(columns=[label])
y_valid = valid_df[label].astype(int)
X_valid = valid_df.drop(columns=[label])

X_train = pd.get_dummies(X_train, drop_first=True)
X_valid = pd.get_dummies(X_valid, drop_first=True)
X_train, X_valid = X_train.align(X_valid, join="left", axis=1, fill_value=0)

rf = RandomForestClassifier(
    n_estimators=800,
    max_depth=20,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
 )
rf.fit(X_train, y_train)

print(f"Training features: {X_train.shape[1]}")
print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_valid)}")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    roc_auc_score, average_precision_score
 )

preds = rf.predict(X_valid)
probs = rf.predict_proba(X_valid)[:, 1]

print("=== Validation metrics (threshold=0.5) ===")
print("Accuracy:", round(accuracy_score(y_valid, preds), 3))
print("Precision:", round(precision_score(y_valid, preds, zero_division=0), 3))
print("Recall:", round(recall_score(y_valid, preds), 3))
print("F1:", round(f1_score(y_valid, preds), 3))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_valid, preds), 3))
print("MCC:", round(matthews_corrcoef(y_valid, preds), 3))
print("Normalized MCC:", round((matthews_corrcoef(y_valid, preds) + 1) / 2, 3))
print("ROC AUC:", round(roc_auc_score(y_valid, probs), 3))
print("PR AUC:", round(average_precision_score(y_valid, probs), 3))

print("\nValidation class distribution:")
print(y_valid.value_counts(dropna=False))